# Baseline Explained — X_train / y_train / X_test / y_test + `.fit()`

Companion to `preprocess_split_train_test.py`. Run all cells top to bottom (~30s). Shows exactly what the baseline sees and predicts.

In [1]:
import pandas as pd
import numpy as np
import preprocess_split_train_test as P
from sklearn.metrics import mean_absolute_error

df = P.load_real_panel(verbose=False)
X, y, city, splits, dates = P.build_feature_matrix(df)

X_train, y_train = X[splits['train']], y[splits['train']]
X_test, y_test = X[splits['test']], y[splits['test']]
city_train, city_test = city[splits['train']], city[splits['test']]
dates_train, dates_test = dates[splits['train']], dates[splits['test']]

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}  (2016-2019)')
print(f'X_test : {X_test.shape}  y_test : {y_test.shape}  (2020-2021)')

  Collapsing 6 duplicate city-year-week rows (median of reported values)
  Cagayan de Oro City: winsorising 2 week(s) above cap 484 (max 998)
  Zamboanga City: winsorising 1 week(s) above cap 513 (max 3382)
Feature matrix: 753 rows x 28 features
  Features: 28 (expected 28)
  Train:      606 rows  (2016–2019)
  Embargo:    3 rows  (skipped)
  Test (2020–2021): 144 rows
X_train: (606, 28)  y_train: (606,)  (2016-2019)
X_test : (144, 28)  y_test : (144,)  (2020-2021)


## 1. `X_train` — what the model sees (features include weather)
Each row = one city-week at forecast origin `t`. Columns = past-only info: case momentum + 12 weather lags + 3 rolling + 7 bioclimatic + 2 seasonality = 28.

In [2]:
display(X_train.head())
print('Feature groups:')
print(' cases:', P.CASE_MOMENTUM)
print(' weather lags:', P.WEATHER_LAGS)
print(' rolling:', P.ROLLING_AGGREGATES)
print(' bioclim:', P.BIOCLIMATIC)
print(' season:', P.SEASONALITY)

# One concrete row, broken down
r = X_train.iloc[0]
print('\nExample row 0:', city_train.iloc[0], dates_train.iloc[0].date())
print(' cases:', r[P.CASE_MOMENTUM].to_dict())
print(' weather sample:', {k: round(float(r[k]),2) for k in ['rainfall_lag1','temperature_lag1','humidity_lag1','rainfall_rollsum4']})
print(' season:', {'sin_week': r['sin_week'], 'cos_week': r['cos_week']})

,rainfall_lag1,rainfall_lag2,rainfall_lag3,rainfall_lag4,temperature_lag1,temperature_lag2,temperature_lag3,temperature_lag4,humidity_lag1,humidity_lag2,...,cases_rollmean4,heat_index,tmin,tmax,diurnal_temp_range,spi3,pnp,ndvi,sin_week,cos_week
0,8.3157,5.4246,2.9876,1.0824,25.5684,25.8873,25.6729,25.5420,77.8371,81.4210,...,54.75,28.1297,23.1555,28.7473,5.5918,-1.5756,31.8087,0.6286,0.5681,0.8230
1,6.1209,8.3157,5.4246,2.9876,25.9648,25.5684,25.8873,25.6729,84.7022,77.8371,...,48.25,27.7148,23.0534,28.5508,5.4974,-1.5756,31.8087,0.6109,0.6631,0.7485
2,10.7074,6.1209,8.3157,5.4246,25.8568,25.9648,25.5684,25.8873,82.4154,84.7022,...,45.75,27.2614,22.1782,28.7247,6.5465,-1.5756,31.8087,0.6049,0.7485,0.6631
3,6.5673,10.7074,6.1209,8.3157,25.5587,25.8568,25.9648,25.5684,81.7598,82.4154,...,40.00,27.6044,22.5526,28.6293,6.0768,-1.5756,31.8087,0.5926,0.8230,0.5681
4,7.8928,6.5673,10.7074,6.1209,25.7360,25.5587,25.8568,25.9648,82.9527,81.7598,...,35.00,29.1555,23.4977,29.3582,5.8604,-2.1786,15.2531,0.5758,0.8855,0.4647


Feature groups:
 cases: ['cases_lag1', 'cases_lag2', 'cases_lag4', 'cases_rollmean4']
 weather lags: ['rainfall_lag1', 'rainfall_lag2', 'rainfall_lag3', 'rainfall_lag4', 'temperature_lag1', 'temperature_lag2', 'temperature_lag3', 'temperature_lag4', 'humidity_lag1', 'humidity_lag2', 'humidity_lag3', 'humidity_lag4']
 rolling: ['rainfall_rollsum4', 'temperature_rollmean4', 'humidity_rollmean4']
 bioclim: ['heat_index', 'tmin', 'tmax', 'diurnal_temp_range', 'spi3', 'pnp', 'ndvi']
 season: ['sin_week', 'cos_week']

Example row 0: Cagayan de Oro City 2016-02-01
 cases: {'cases_lag1': 54.0, 'cases_lag2': 55.0, 'cases_lag4': 64.0, 'cases_rollmean4': 54.75}
 weather sample: {'rainfall_lag1': 8.32, 'temperature_lag1': 25.57, 'humidity_lag1': 77.84, 'rainfall_rollsum4': 17.81}
 season: {'sin_week': np.float64(0.5681), 'cos_week': np.float64(0.823)}


## 2. `y_train` — the target variable
`y_train[i]` = actual dengue cases **2 weeks after** `X_train[i]` (column `target_cases_t2`). The model must map present → future.

In [3]:
demo_train = pd.DataFrame({'date_t': pd.to_datetime(dates_train.values), 'city': city_train.values, 'cases_now_lag1': X_train['cases_lag1'].values, 'y_train_cases_t2': y_train.values})
display(demo_train.head(10))
print('Alignment check: y_train[0] is the case count 2 weeks after X_train row 0.')

,date_t,city,cases_now_lag1,y_train_cases_t2
0,2016-02-01,Cagayan de Oro City,54.0,32.0
1,2016-02-08,Cagayan de Oro City,38.0,34.0
2,2016-02-15,Cagayan de Oro City,36.0,37.0
3,2016-02-22,Cagayan de Oro City,32.0,24.0
4,2016-02-29,Cagayan de Oro City,34.0,16.0
5,2016-03-07,Cagayan de Oro City,37.0,28.0
6,2016-03-14,Cagayan de Oro City,24.0,11.0
7,2016-03-21,Cagayan de Oro City,16.0,12.0
8,2016-03-28,Cagayan de Oro City,28.0,19.0
9,2016-04-04,Cagayan de Oro City,11.0,16.0


Alignment check: y_train[0] is the case count 2 weeks after X_train row 0.


## 3. `X_test` / `y_test` — same format, but future period (2020-2021, never seen in training)
Identical 28 columns. Only difference = dates are later. This is what we evaluate on.

In [4]:
display(X_test.head())
demo_test = pd.DataFrame({'date_t': pd.to_datetime(dates_test.values), 'city': city_test.values, 'cases_now_lag1': X_test['cases_lag1'].values, 'y_test_cases_t2': y_test.values})
display(demo_test.head(10))
print(f'X_test rows: {len(X_test)}, y_test rows: {len(y_test)}')

,rainfall_lag1,rainfall_lag2,rainfall_lag3,rainfall_lag4,temperature_lag1,temperature_lag2,temperature_lag3,temperature_lag4,humidity_lag1,humidity_lag2,...,cases_rollmean4,heat_index,tmin,tmax,diurnal_temp_range,spi3,pnp,ndvi,sin_week,cos_week
203,10.6936,6.7600,6.1984,30.7676,25.7049,25.8604,25.7211,25.4122,83.9491,80.5843,...,43.75,28.0172,22.8901,29.1127,6.2226,-1.3498,20.0734,0.6935,0.3546,0.9350
204,10.0133,10.6936,6.7600,6.1984,25.7538,25.7049,25.8604,25.7211,86.0532,83.9491,...,51.00,27.5083,23.2427,28.1838,4.9412,-1.3498,20.0734,0.6838,0.4647,0.8855
205,0.5505,10.0133,10.6936,6.7600,25.6045,25.7538,25.7049,25.8604,84.2478,86.0532,...,52.00,27.8466,22.9860,28.5536,5.5676,-1.3707,25.2488,0.6756,0.5681,0.8230
206,0.0000,0.5505,10.0133,10.6936,25.7092,25.6045,25.7538,25.7049,85.2851,84.2478,...,46.75,26.8474,22.3000,28.0913,5.7913,-1.4229,38.1872,0.6684,0.6631,0.7485
207,24.5587,0.0000,0.5505,10.0133,25.2166,25.7092,25.6045,25.7538,83.5020,85.2851,...,45.50,26.9203,22.6348,27.9931,5.3583,-1.4229,38.1872,0.6544,0.7485,0.6631


,date_t,city,cases_now_lag1,y_test_cases_t2
0,2020-01-13,Cagayan de Oro City,45.0,44.0
1,2020-01-20,Cagayan de Oro City,48.0,40.0
2,2020-01-27,Cagayan de Oro City,50.0,9.0
3,2020-02-03,Cagayan de Oro City,44.0,29.0
4,2020-02-10,Cagayan de Oro City,40.0,31.0
5,2020-02-17,Cagayan de Oro City,9.0,23.0
6,2020-02-24,Cagayan de Oro City,29.0,18.0
7,2020-03-02,Cagayan de Oro City,31.0,13.0
8,2020-03-09,Cagayan de Oro City,23.0,7.0
9,2020-03-16,Cagayan de Oro City,18.0,12.0


X_test rows: 144, y_test rows: 144


## 4. Baseline predicts: `y_hat = cases_lag1` (no learning, no `.fit`)
Formula in `preprocess_split_train_test.py:658`: `persistence_predict(X) = X['cases_lag1']`. It just repeats the most recent week.

In [5]:
from preprocess_split_train_test import persistence_predict
pred_train_base = persistence_predict(X_train)
pred_test_base = persistence_predict(X_test)

show_train = pd.DataFrame({'city': city_train.values[:8], 'date_t': pd.to_datetime(dates_train.values[:8]), 'X_cases_lag1': X_train['cases_lag1'].values[:8], 'baseline_pred': pred_train_base[:8].round(1), 'y_train_true': y_train.values[:8]})
show_train['error'] = (show_train['y_train_true'] - show_train['baseline_pred']).round(1)
display(show_train)
print(f'Baseline train MAE: {mean_absolute_error(y_train, pred_train_base):.1f}')

show_test = pd.DataFrame({'city': city_test.values[:8], 'date_t': pd.to_datetime(dates_test.values[:8]), 'X_cases_lag1': X_test['cases_lag1'].values[:8], 'baseline_pred': pred_test_base[:8].round(1), 'y_test_true': y_test.values[:8]})
show_test['error'] = (show_test['y_test_true'] - show_test['baseline_pred']).round(1)
display(show_test)
print(f'Baseline test MAE: {mean_absolute_error(y_test, pred_test_base):.1f}')

,city,date_t,X_cases_lag1,baseline_pred,y_train_true,error
0,Cagayan de Oro City,2016-02-01,54.0,54.0,32.0,-22.0
1,Cagayan de Oro City,2016-02-08,38.0,38.0,34.0,-4.0
2,Cagayan de Oro City,2016-02-15,36.0,36.0,37.0,1.0
3,Cagayan de Oro City,2016-02-22,32.0,32.0,24.0,-8.0
4,Cagayan de Oro City,2016-02-29,34.0,34.0,16.0,-18.0
5,Cagayan de Oro City,2016-03-07,37.0,37.0,28.0,-9.0
6,Cagayan de Oro City,2016-03-14,24.0,24.0,11.0,-13.0
7,Cagayan de Oro City,2016-03-21,16.0,16.0,12.0,-4.0


Baseline train MAE: 26.2


,city,date_t,X_cases_lag1,baseline_pred,y_test_true,error
0,Cagayan de Oro City,2020-01-13,45.0,45.0,44.0,-1.0
1,Cagayan de Oro City,2020-01-20,48.0,48.0,40.0,-8.0
2,Cagayan de Oro City,2020-01-27,50.0,50.0,9.0,-41.0
3,Cagayan de Oro City,2020-02-03,44.0,44.0,29.0,-15.0
4,Cagayan de Oro City,2020-02-10,40.0,40.0,31.0,-9.0
5,Cagayan de Oro City,2020-02-17,9.0,9.0,23.0,14.0
6,Cagayan de Oro City,2020-02-24,29.0,29.0,18.0,-11.0
7,Cagayan de Oro City,2020-03-02,31.0,31.0,13.0,-18.0


Baseline test MAE: 6.0


## 5. `.fit(X_train, y_train)` — baseline has NONE, XGB learns the residual
- Baseline: `build_model('persistence')` returns `None`. There is nothing to fit.
- XGB: `model.fit(X_train, y_train)` learns `residual = y_train - cases_lag1` from all 28 (or 6) features, then predicts `max(0, cases_lag1 + shrinkage * XGB_residual)`.

In [6]:
from preprocess_split_train_test import build_model
m = build_model('persistence')
print(f"build_model('persistence') -> {m}  (no .fit, predict = X['cases_lag1'])")

# XGB side, fit on the SAME X_train / y_train
xgb = build_model('xgb_full')
print(f"build_model('xgb_full') -> {type(xgb).__name__} with {len(xgb.feature_set)} features")
xgb.fit(X_train, y_train)  # <-- the .fit(X_train, y_train)
print('fitted on train only. Now compare on 5 TEST rows:')
px = xgb.predict(X_test.iloc[:5])
comp = pd.DataFrame({'city': city_test.values[:5], 'baseline_pred': persistence_predict(X_test.iloc[:5]).round(1), 'xgb_full_pred': np.round(px,1), 'y_test_true': y_test.values[:5]})
display(comp)
print(f'Full-test MAE: baseline {mean_absolute_error(y_test, persistence_predict(X_test)):.1f} vs xgb_full {mean_absolute_error(y_test, xgb.predict(X_test)):.1f}')

build_model('persistence') -> None  (no .fit, predict = X['cases_lag1'])
build_model('xgb_full') -> ResidualXGBRegressor with 28 features


fitted on train only. Now compare on 5 TEST rows:


,city,baseline_pred,xgb_full_pred,y_test_true
0,Cagayan de Oro City,45.0,44.2,44.0
1,Cagayan de Oro City,48.0,46.0,40.0
2,Cagayan de Oro City,50.0,48.1,9.0
3,Cagayan de Oro City,44.0,43.0,29.0
4,Cagayan de Oro City,40.0,39.1,31.0


Full-test MAE: baseline 6.0 vs xgb_full 5.8


## Takeaway
- `X_train` (606x28, 2016-2019) + `y_train` (cases at t+2) = what models learn from.
- `X_test` (144x28, 2020-2021) + `y_test` = what they are graded on.
- Baseline prediction = copy `cases_lag1` column, needs no `.fit`.
- XGB prediction = `cases_lag1 + shrinkage * XGB(X)` where `XGB.fit(X_train, y_train-cases_lag1)` learns the correction from weather + case history.